# 딥러닝 따라하기2_분류

# 1.환경준비

* 라이브러리 Import

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 2.Classification : 대학원 지원

## (1) 데이터 전처리

### 1) 데이터 준비

In [2]:
path = "https://raw.githubusercontent.com/DA4BAM/dataset/master/Graduate_apply.csv"
data = pd.read_csv(path)
data.head()

,admit,gre,gpa,rank
0,0,380,3.61,3
1,1,660,3.67,3
2,1,800,4.00,1
3,1,640,3.19,4
4,0,520,2.93,4


In [3]:
target = 'admit'
x = data.drop(target, axis=1)
y = data.loc[:, target]

### 2) 가변수화

### 3) 데이터분할

In [4]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.2, random_state = 20)

## (2) 머신러닝 모델링 : 로지스틱회귀

In [5]:
# 필요한 알고리즘을 불러 옵시다.
from sklearn.linear_model import LogisticRegression

### 1) 모델 선언

In [6]:
model = LogisticRegression()

### 2) 학습

In [7]:
model.fit(x_train, y_train)

LogisticRegression()

### 3) 예측

In [8]:
pred = model.predict(x_val)

### 4) 검증
만든 모델은 얼마나 정확한지 검증해 봅시다.



In [9]:
print(confusion_matrix(y_val, pred))
print('-'*50)
print(classification_report(y_val, pred))

[[48  7]
 [18  7]]
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.73      0.87      0.79        55
           1       0.50      0.28      0.36        25

    accuracy                           0.69        80
   macro avg       0.61      0.58      0.58        80
weighted avg       0.66      0.69      0.66        80



## (3) 딥러닝 모델링
* 필요한 함수들 불러오기
* 모델 선언
* 학습
* 예측
* 성능 검증

### 1) 전처리 : Scaling

In [10]:
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

### 2) 필요한 함수 불러오기

In [11]:
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.backend import clear_session

### 3) 모델 선언

In [12]:
nfeatures = x_train.shape[1] #num of columns
nfeatures

3

In [13]:
# 메모리 정리
clear_session()

# Sequential 타입 모델 선언
model = Sequential([Input(shape = (nfeatures,)),
                    Dense(1, activation = 'sigmoid' ) ])  # 회귀에서는 LeRU(relu)를 사용했었음

# 모델요약
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4 (16.00 B)

 Trainable params: 4 (16.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
# 컴파일
model.compile(optimizer='adam', loss= 'binary_crossentropy')

### 4) 학습

In [15]:
model.fit(x_train, y_train)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6999  


### 5) 예측

In [16]:
pred = model.predict(x_val)
pred = np.where(pred>= 0.5, 1, 0)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


### 6) 검증
만든 모델은 얼마나 정확한지 검증해 봅시다.



In [17]:
print(confusion_matrix(y_val, pred))
print('-'*50)
print(classification_report(y_val, pred))

[[44 11]
 [23  2]]
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.66      0.80      0.72        55
           1       0.15      0.08      0.11        25

    accuracy                           0.57        80
   macro avg       0.41      0.44      0.41        80
weighted avg       0.50      0.57      0.53        80



# 3.Classification : mobile

## (1) 데이터 전처리

### 1) 데이터 준비

In [18]:
path = "https://raw.githubusercontent.com/DA4BAM/dataset/master/mobile_churn_simple.csv"
data = pd.read_csv(path)
data['CHURN'] = data['CHURN'].map({'STAY':0, 'LEAVE':1})
data.head()

,INCOME,OVERAGE,LEFTOVER,HOUSE,HANDSET_PRICE,OVER_15MINS_CALLS_PER_MONTH,AVERAGE_CALL_DURATION,CHURN
0,31953,0,6,313378,161,0,4,0
1,36147,0,13,800586,244,0,6,0
2,27273,230,0,305049,201,16,15,0
3,120070,38,33,788235,780,3,2,1
4,29215,208,85,224784,241,21,1,0


|	구분	|	변수 명	|	내용	|	type	|	비고	|
|	----	|	----	|	----	|	----	|	----	|
|	**Target**	|	**CHURN**	|	이탈여부	|	범주	| 0,1	|
|	feature	|	INCOME	|	소득수준(달러)	|	숫자	|		|
|	feature	|	OVERAGE	|	월평균 초과사용시간(분)	|	숫자	| |
|	feature	|	LEFTOVER	|	월평균 잔여시간(%)	|	숫자	| 	|
|	feature	|	HOUSE	|	집가격(달러)	|	숫자	|	|
|	feature	|	HANDSET_PRICE	|	휴대폰가격(달러)	|	숫자	|		|
|	feature	|	OVER_15MINS_CALLS_PER_MONTH	|	월평균 장기통화 횟수	|	숫자	| 		|
|	feature	|	AVERAGE_CALL_DURATION	|	평균통화시간(분)	|	숫자	|		|

In [19]:
target = 'CHURN'
x = data.drop(target, axis=1)
y = data.loc[:, target]

### 2) 가변수화

### 3) 데이터분할

In [20]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.2, random_state = 20)

### 4) Scaling

In [21]:
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

## (2) 모델링
* 필요한 함수들 불러오기
* 모델 선언
* 학습
* 예측
* 성능 검증

In [22]:
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.backend import clear_session

### 1) 모델 선언

In [23]:
nfeatures = x_train.shape[1] #num of columns
nfeatures

7

In [25]:
# 메모리 정리
clear_session()

# Sequential 타입 모델 선언
model = Sequential([Input(shape=(nfeatures, )),
                    Dense(1, activation='sigmoid')])

# 모델요약
model.summary

<bound method Model.summary of <Sequential name=sequential, built=True>>

In [28]:
# 컴파일
model.compile(optimizer='adam', loss='binary_crossentropy')

### 2) 학습

In [29]:
model.fit(x_train, y_train)

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7140


### 3) 예측

In [30]:
pred = model.predict(x_val)
pred = np.where(pred>=0.5, 1, 0)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


### 4) 검증
만든 모델은 얼마나 정확한지 검증해 봅시다.



In [31]:
print(confusion_matrix(y_val, pred))
print('-'*50)
print(classification_report(y_val, pred))

[[1121  917]
 [ 909 1053]]
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.55      0.55      0.55      2038
           1       0.53      0.54      0.54      1962

    accuracy                           0.54      4000
   macro avg       0.54      0.54      0.54      4000
weighted avg       0.54      0.54      0.54      4000

